In [0]:
%sql

LIST '/Volumes/tiktok_data_eng/landing/archivos/';

In [0]:
%sql
-- Ver que contiene el json
select *
from read_files(
    '/Volumes/tiktok_data_eng/landing/archivos/tiktok_raw_20260825_011510.json',
    format => 'json',
    header => true
)

In [0]:
%sql
-- ver tipo de columnas que viene 
DESCRIBE QUERY
select *
from read_files(
    '/Volumes/tiktok_data_eng/landing/archivos/tiktok_raw_20260825_011510.json',
    format => 'json'
)

In [0]:
import json

ruta = "/Volumes/tiktok_data_eng/landing/archivos/tiktok_raw_20260825_011510.json"

with open(ruta, "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

print("Cantidad de registros:", len(datos))

print("\nPrimer registro:")
print(json.dumps(datos[0], indent=2, ensure_ascii=False))

In [0]:
# ============================================================
# BRONZE - VISUALIZACIÓN DE DATOS RAW
# ============================================================
# Objetivo:
# - Leer el JSON que ya tenemos en Landing
# - NO volver a consultar Apify
# - NO transformar los datos
# - NO persistir todavía
# - Convertir todos los campos a STRING
# - Mostrar los 166 registros como tabla
# ============================================================

import json

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)


# ============================================================
# 1. RUTA DEL ARCHIVO RAW
# ============================================================

RUTA_RAW = (
    "/Volumes/tiktok_data_eng/"
    "landing/archivos/"
    "tiktok_raw_20260825_011510.json"
)


# ============================================================
# 2. LEER EL JSON
# ============================================================

with open(RUTA_RAW, "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)


print("=" * 70)
print("DATOS RAW")
print("=" * 70)

print(f"Cantidad de registros: {len(datos)}")


# ============================================================
# 3. OBTENER TODAS LAS COLUMNAS
# ============================================================
# Algunos registros podrían tener campos que otros no tengan.
# Por eso obtenemos la unión de todos los campos disponibles.

columnas = set()

for registro in datos:
    columnas.update(registro.keys())

columnas = sorted(columnas)

print(f"Cantidad de columnas: {len(columnas)}")

print("\nColumnas encontradas:")
for columna in columnas:
    print(f" - {columna}")


# ============================================================
# 4. CREAR ESQUEMA
# ============================================================
# IMPORTANTE:
# Todo será STRING.
#
# Incluso objetos como:
#
# stats
# author
# music
#
# y listas como:
#
# hashtags
# mentions
#
# se conservarán como JSON dentro de un STRING.
# ============================================================

schema = StructType(
    [
        StructField(
            columna,
            StringType(),
            True
        )
        for columna in columnas
    ]
)


# ============================================================
# 5. CONSTRUIR LAS FILAS
# ============================================================

filas = []

for registro in datos:

    fila = []

    for columna in columnas:

        valor = registro.get(columna)

        # ----------------------------------------------------
        # Campo inexistente o NULL
        # ----------------------------------------------------

        if valor is None:

            fila.append(None)

        # ----------------------------------------------------
        # Diccionarios o listas
        # ----------------------------------------------------
        # Los conservamos como JSON dentro del STRING.
        #
        # Ejemplo:
        #
        # stats:
        # {"plays":1270,"likes":76,...}
        #
        # hashtags:
        # ["python","dataengineering"]
        # ----------------------------------------------------

        elif isinstance(valor, (dict, list)):

            fila.append(
                json.dumps(
                    valor,
                    ensure_ascii=False
                )
            )

        # ----------------------------------------------------
        # Cualquier otro tipo
        # ----------------------------------------------------

        else:

            fila.append(
                str(valor)
            )

    filas.append(tuple(fila))


# ============================================================
# 6. CREAR DATAFRAME BRONZE
# ============================================================

tiktok_raw = spark.createDataFrame(
    filas,
    schema
)


# ============================================================
# 7. MOSTRAR ESQUEMA
# ============================================================

print("\n" + "=" * 70)
print("ESQUEMA BRONZE")
print("=" * 70)

tiktok_raw.printSchema()

tiktok_raw.write.format('Delta').mode('overwrite').saveAsTable('tiktok_data_eng.bronze.tiktok_bronze')


# ============================================================
# 8. MOSTRAR DATOS
# ============================================================

print("\n" + "=" * 70)
print("DATOS BRONZE")
print("=" * 70)

display(tiktok_raw)

In [0]:
%sql
select *
from tiktok_data_eng.bronze.tiktok_bronze

In [0]:
%sql
-- EDA inicial
-- conteo de registros 

select count(*) as total_registros 
from tiktok_data_eng.bronze.tiktok_bronze

In [0]:
%sql
-- muestra de datos para verificar datos
select *
from tiktok_data_eng.bronze.tiktok_bronze
limit(5);